# ECG-FM — Colab Training Notebook

**Before running:** Runtime → Change runtime type → **T4 GPU**

## Stage 2 Fine-tuning (cells 1–8)
- **First time only:** Cells 1 → 2 → 3 → 4 (downloads PTB-XL to Drive, ~3 hrs)
- **Every session:** Cells 1 → 2 → **5** → 6 → 7 → 8
- Cell 4 uses `wget -nc` — safe to re-run

## Multilabel Backbone (cells 9–12)
- Run **after** Stage 2 is complete and ecgfm_stage2.pt is on Drive
- Upload `ecgfm_multilabel.py` and `multilabel_classifier.py` to Drive first
- Cells 9 → 10 (precompute, ~15 min) → 11 (train head) → 12 (save)


In [ ]:
# Cell 1 â€” Check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU â€” Runtime â†’ Change runtime type â†’ T4 GPU')
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import psutil
print('RAM :', round(psutil.virtual_memory().total / 1e9, 1), 'GB')

In [ ]:
# Cell 2 â€” Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
MODEL_DIR = '/content/drive/MyDrive/EKG_models'
os.makedirs(MODEL_DIR, exist_ok=True)

for f in ['mimic_iv_ecg_physionet_pretrained.pt', 'ecgfm_stage1.pt']:
    status = 'OK' if os.path.exists(f'{MODEL_DIR}/{f}') else 'MISSING'
    print(f'  {status} â€” {f}')

In [ ]:
# Cell 3 â€” Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb', 'scipy', 'scikit-learn'], check=True)
print('Dependencies ready')

In [ ]:
# Cell 4 â€” Download PTB-XL 500Hz to Drive (FIRST TIME ONLY â€” takes ~3 hrs)
# Safe to re-run: wget -nc skips files already on Drive.
# If disconnected mid-download: reconnect, re-run cells 1-2, then re-run this cell.

import os
from pathlib import Path

MODEL_DIR    = '/content/drive/MyDrive/EKG_models'
DRIVE_PTBXL  = f'{MODEL_DIR}/ptbxl'
os.makedirs(DRIVE_PTBXL, exist_ok=True)

for fname in ['ptbxl_database.csv', 'scp_statements.csv']:
    if not os.path.exists(f'{DRIVE_PTBXL}/{fname}'):
        os.system(f'wget -q -nc -P {DRIVE_PTBXL} https://physionet.org/files/ptb-xl/1.0.3/{fname}')
    print(f'  {"OK" if os.path.exists(f"{DRIVE_PTBXL}/{fname}") else "MISSING"} â€” {fname}')

rec_dir = Path(f'{DRIVE_PTBXL}/records500')
n_dat = len(list(rec_dir.rglob('*.dat'))) if rec_dir.exists() else 0
print(f'\nSignal files on Drive: {n_dat} / 21837')

if n_dat < 21000:
    print('Downloading records500 to Drive (skipping files already present)...')
    os.system(f'wget -r -nc -np -nH --cut-dirs=3 -P {DRIVE_PTBXL} https://physionet.org/files/ptb-xl/1.0.3/records500/')
    n_dat = len(list(rec_dir.rglob('*.dat')))
    print(f'After download: {n_dat} .dat files on Drive')
else:
    print('All records already on Drive â€” skipping download')

print('\nCell 4 done. Run Cell 5 to set up the session, then Cell 6 to train.')

In [ ]:
# Cell 5 â€” Session setup (run this EVERY session, including after reconnect)
# Sets up: training scripts, model weights, PTB-XL symlink, working directory

import os, shutil
os.chdir('/content')

MODEL_DIR   = '/content/drive/MyDrive/EKG_models'
DRIVE_PTBXL = f'{MODEL_DIR}/ptbxl'

# â”€â”€ 1. Training scripts from Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
for f in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py']:
    shutil.copy(f'{MODEL_DIR}/{f}', f'/content/{f}')

# â”€â”€ 2. Model weights â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
os.makedirs('/content/models/ecgfm', exist_ok=True)
shutil.copy(f'{MODEL_DIR}/mimic_iv_ecg_physionet_pretrained.pt',
            '/content/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_DIR}/ecgfm_stage1.pt', '/content/models/ecgfm_stage1.pt')

# â”€â”€ 3. PTB-XL symlink â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
os.makedirs('/content/ekg_datasets', exist_ok=True)
link = '/content/ekg_datasets/ptbxl'
if os.path.islink(link): os.remove(link)
elif os.path.exists(link): shutil.rmtree(link)
os.symlink(DRIVE_PTBXL, link)

# â”€â”€ 4. Verify â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
checks = {
    'ecgfm_finetune.py':                     os.path.exists('/content/ecgfm_finetune.py'),
    'models/ecgfm_stage1.pt':                os.path.exists('/content/models/ecgfm_stage1.pt'),
    'models/ecgfm/pretrained encoder':       os.path.exists('/content/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt'),
    'ekg_datasets/ptbxl/ptbxl_database.csv': os.path.exists('/content/ekg_datasets/ptbxl/ptbxl_database.csv'),
    'signal files (sample)':                 os.path.exists('/content/ekg_datasets/ptbxl/records500/00000/00001_hr.dat'),
}
all_ok = True
for k, v in checks.items():
    print(f'  {"OK" if v else "MISSING"} â€” {k}')
    if not v: all_ok = False

print()
if all_ok:
    print('All OK â€” run Cell 6 (keep-alive) then Cell 7 to train')
else:
    print('Fix MISSING items before training')

In [ ]:
# Cell 6 â€” Keep-alive (run before Cell 7 to prevent idle disconnect)
import time, threading

def _keep_alive():
    i = 0
    while True:
        time.sleep(30)
        i += 1
        print(f'  [keep-alive {i}]', flush=True)

t = threading.Thread(target=_keep_alive, daemon=True)
t.start()
print('Keep-alive started (prints every 30s)')

In [ ]:
# Cell 7 â€” Train Stage 2
# ~20 min/epoch on T4, ~3 hrs total. Saves models/ecgfm_stage2.pt on each improvement.
# OOM? change --batch_s2 8 to --batch_s2 4

import os
os.chdir('/content')
!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

In [ ]:
# Cell 8 â€” Save result to Drive
import os, shutil, torch

MODEL_DIR = '/content/drive/MyDrive/EKG_models'

if os.path.exists('/content/models/ecgfm_stage2.pt'):
    dest = f'{MODEL_DIR}/ecgfm_stage2.pt'
    shutil.copy('/content/models/ecgfm_stage2.pt', dest)
    print(f'Saved to Drive ({round(os.path.getsize(dest)/1e6)} MB)')

    ckpt = torch.load('/content/models/ecgfm_stage2.pt', map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print(f'\nFinal test results:')
    print(f'  Accuracy  : {tm.get("acc", 0):.1%}')
    print(f'  HYP F1    : {tm.get("hyp_f1", 0):.3f}  (baseline 0.375)')
    print(f'  Macro F1  : {tm.get("macro_f1", 0):.3f}  (baseline 0.492)')
else:
    print('WARNING: ecgfm_stage2.pt not found')

## ECG-FM Multilabel Backbone

Use the frozen Stage 2 encoder as a backbone for 12-class multi-label classification.
- **Cell 9** — Session setup (copies multilabel scripts, Stage 2 checkpoint)
- **Cell 10** — Precompute 768-dim embeddings for all PTB-XL records (~15 min on T4)
- **Cell 11** — Train MLP head (fast, runs on CPU too)
- **Cell 12** — Save to Drive


In [ ]:
# Cell 9 - Multilabel session setup
# Run every session after Cells 1-2-3
import os, re, shutil, subprocess, sys
os.chdir('/content')

MODEL_DIR = '/content/drive/MyDrive/EKG_models'

# 1. Copy scripts from Drive
for fname in ['ecgfm_multilabel.py', 'multilabel_classifier.py',
              'ecgfm_encoder.py', 'cnn_classifier.py']:
    src = f'{MODEL_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{fname}')
        print(f'  OK: {fname}')
    else:
        print(f'  MISSING: {fname}')

# 2. Fix import names (safety patch)
ml_path = '/content/ecgfm_multilabel.py'
with open(ml_path) as f:
    code = f.read()
code = re.sub(r'load_multilabel_data\w*', 'load_multilabel_dataset', code)
code = re.sub(r'build_pos_weight', 'compute_pos_weights', code)
with open(ml_path, 'w') as f:
    f.write(code)
print('  Patched: imports')

# 3. Model checkpoints
os.makedirs('/content/models/ecgfm', exist_ok=True)

# Stage 2 fine-tuned weights
stage2_src = f'{MODEL_DIR}/ecgfm_stage2.pt'
stage2_dst = '/content/models/ecgfm_stage2.pt'
if os.path.exists(stage2_src):
    if not os.path.exists(stage2_dst):
        os.symlink(stage2_src, stage2_dst)
    print('  OK: ecgfm_stage2.pt')
else:
    print('  MISSING: ecgfm_stage2.pt')

# Original pretrained (needed for architecture/config)
pretrained_src = f'{MODEL_DIR}/mimic_iv_ecg_physionet_pretrained.pt'
pretrained_dst = '/content/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt'
if os.path.exists(pretrained_src):
    if not os.path.exists(pretrained_dst):
        os.symlink(pretrained_src, pretrained_dst)
    print('  OK: mimic_iv_ecg_physionet_pretrained.pt')
else:
    print('  MISSING: mimic_iv_ecg_physionet_pretrained.pt')

# 4. Copy ptbxl to local SSD (10x faster reads vs Drive)
local_ptbxl = '/content/ptbxl_local'
link = '/content/ekg_datasets/ptbxl'
if os.path.exists(f'{local_ptbxl}/ptbxl_database.csv'):
    from pathlib import Path
    n = len(list(Path(local_ptbxl).rglob('*.dat')))
    print(f'  OK: ptbxl already on local SSD ({n} .dat files)')
else:
    print('  Copying ptbxl to local SSD (~3-5 min)...')
    os.makedirs(local_ptbxl, exist_ok=True)
    os.system(f'cp {MODEL_DIR}/ptbxl/ptbxl_database.csv {local_ptbxl}/')
    os.system(f'cp {MODEL_DIR}/ptbxl/scp_statements.csv {local_ptbxl}/')
    os.system(f'cp -r {MODEL_DIR}/ptbxl/records500 {local_ptbxl}/')
    from pathlib import Path
    n = len(list(Path(local_ptbxl).rglob('*.dat')))
    print(f'  OK: ptbxl copied ({n} .dat files)')

os.makedirs('/content/ekg_datasets', exist_ok=True)
if os.path.islink(link): os.remove(link)
elif os.path.exists(link): shutil.rmtree(link)
os.symlink(local_ptbxl, link)
print('  OK: ptbxl -> local SSD')

# 5. Verify imports
r = subprocess.run([sys.executable, '-c', 'import ecgfm_multilabel'], capture_output=True, text=True)
if r.returncode == 0:
    print()
    print('All OK - run Cell 10')
else:
    print('Import ERROR:', r.stderr[-400:])


In [ ]:
# Cell 10 - Precompute ECG-FM embeddings
# Reads from local SSD, should take ~15 min on T4
# Progress prints every 500 records during aux build, every 400 during encode:
#   aux 500/6465
#   aux 1000/6465
#   ...
#   Precomputing 6465 embeddings...
#   400/6465  45 rec/s  ETA 12 min
#   800/6465  48 rec/s  ETA 9 min
#   ...
#   Done in 850s  -> saves models/ecgfm_ml_embeddings.npz
import os
os.chdir('/content')
!python -u ecgfm_multilabel.py --precompute


In [ ]:
# Cell 11 - Train MLP head (60 epochs, ~2 min)
# Architecture: Linear(782->256) -> GELU -> Dropout(0.3) -> Linear(256->12)
# Saves models/ecgfm_ml_head.pt on each MacroF1 improvement
# Baseline to beat: MacroF1=0.699, MacroAUROC=0.972
import os
os.chdir('/content')
!python -u ecgfm_multilabel.py --train
!python -u ecgfm_multilabel.py --eval


In [ ]:
# Cell 12 - Save multilabel results to Drive
import os, shutil

MODEL_DIR = '/content/drive/MyDrive/EKG_models'

for fname in ['ecgfm_ml_embeddings.npz', 'ecgfm_ml_head.pt']:
    src = f'/content/models/{fname}'
    if os.path.exists(src):
        dest = f'{MODEL_DIR}/{fname}'
        shutil.copy(src, dest)
        size_mb = round(os.path.getsize(dest)/1e6, 1)
        print(f'  Saved {fname} ({size_mb} MB) -> Drive')
    else:
        print(f'  MISSING: {fname} -- run Cell 10/11 first')

print('Done! Download ecgfm_ml_head.pt from Drive to local models/ folder')
